In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/annotated/checkpoints/post_cell_4.pickle

trying: ['load_part']
me:  1
trying: ['load_supplier']
me:  5
trying: ['pd']
me:  0
trying: ['partsupp']
me:  3
trying: ['load_partsupp']
me:  3
trying: ['orig_output']
me:  2
trying: ['load_nation']
me:  7
trying: ['region']
me:  9
trying: ['nation']
me:  7
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['supplier']
me:  5
trying: ['load_region']
me:  9
trying: ['part']


me:  1
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable load_part
Declaring variable part
Declaring variable orig_output
Declaring variable partsupp
Declaring variable load_partsupp
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_nation
Declaring variable nation
Declaring variable region
Declaring variable load_region


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# Early filter parts of size 15 and type ending in "BRASS"
part_filtered = (
    part
    [(part["P_SIZE"] == 15) & part["P_TYPE"].str.endswith("BRASS")]
    [["P_PARTKEY", "P_MFGR"]]
)

# Pre-filter partsupp by those part keys to shrink that join
partsupp_filtered = (
    partsupp
    [partsupp["PS_PARTKEY"].isin(part_filtered["P_PARTKEY"]))]
    [["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]
)

# Merge nation→region to get only European nations
r_n = (
    nation[["N_NATIONKEY", "N_NAME", "N_REGIONKEY"]]
    .merge(
        region[region["R_NAME"] == "EUROPE"][["R_REGIONKEY"]],
        left_on="N_REGIONKEY", right_on="R_REGIONKEY",
        how="inner"
    )
    [["N_NATIONKEY", "N_NAME"]]
)

# Merge with supplier
s_r_n = (
    r_n
    .merge(
        supplier[[
            "S_SUPPKEY", "S_NAME", "S_ADDRESS",
            "S_NATIONKEY", "S_PHONE", "S_ACCTBAL", "S_COMMENT"
        ]],
        left_on="N_NATIONKEY", right_on="S_NATIONKEY",
        how="inner"
    )
    [[
        "N_NAME", "S_SUPPKEY", "S_NAME",
        "S_ADDRESS", "S_PHONE", "S_ACCTBAL", "S_COMMENT"
    ]]
)

# Merge with the pre-filtered partsupp
ps_s_r_n = s_r_n.merge(
    partsupp_filtered,
    left_on="S_SUPPKEY", right_on="PS_SUPPKEY",
    how="inner"
)

# Merge in only the filtered parts
merged_df = ps_s_r_n.merge(
    part_filtered,
    left_on="PS_PARTKEY", right_on="P_PARTKEY",
    how="inner"
)

# Compute the minimum supply cost per part
min_vals = (
    merged_df
    .groupby("P_PARTKEY", as_index=False, sort=False)["PS_SUPPLYCOST"]
    .min()
)
min_vals.columns = ["P_PARTKEY_CPY", "MIN_SUPPLYCOST"]

# Keep only those rows whose supply cost equals the per-part minimum
total = (
    merged_df
    .merge(
        min_vals,
        left_on=["P_PARTKEY", "PS_SUPPLYCOST"],
        right_on=["P_PARTKEY_CPY", "MIN_SUPPLYCOST"],
        how="inner"
    )
    [[
        "S_ACCTBAL", "S_NAME", "N_NAME",
        "P_PARTKEY", "P_MFGR", "S_ADDRESS",
        "S_PHONE", "S_COMMENT"
    ]]
)

# Final sort
total = total.sort_values(
    by=["S_ACCTBAL", "N_NAME", "S_NAME", "P_PARTKEY"],
    ascending=[False, True, True, True]
)

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (<unknown>, line 13)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q02/rewritten/o4_mini_high/checkpoints/post_cell_5_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q02/opt_cell_exec_info_5_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [ ]:
opt_output = Out.get(4)